In [ ]:
import os
import numpy as np
import pandas as pd

# ১. পাথ সেটআপ
LABEL_DIR = '/kaggle/input/datasets/mubassirrounak/lmvd-feature/label'
AUDIO_DIR = '/kaggle/input/datasets/mubassirrounak/lmvd-feature/LMVD_Feature/Audio_feature'
VISUAL_DIR = '/kaggle/input/datasets/mubassirrounak/lmvd-feature/LMVD_Feature/Video_feature'

# ২. লেবেল পড়া (দ্রুত উপায়)
all_labels = []
for file in os.listdir(LABEL_DIR):
    if file.endswith(".csv"):
        v_id = file.split('_')[0]
        f_path = os.path.join(LABEL_DIR, file)
        # শুধুমাত্র প্রথম ভ্যালু পড়া, পুরো ফাইল না
        with open(f_path, 'r') as f:
            val = int(f.read().strip().split(',')[0])
        all_labels.append({'video_id': v_id, 'label': val})

labels_df = pd.DataFrame(all_labels)
print(f"Total Labels: {len(labels_df)}")

# ৩. র‍্যাম বাঁচিয়ে ফিচার লোড করার ফাংশন
def load_lite(df, a_dir, v_dir):
    a_feats, v_feats, y = [], [], []
    
    # লিমিট সেট করছি প্রথম ৫০০টি স্যাম্পলের জন্য (যাতে ক্র্যাশ না করে)
    # আপনি চাইলে ৫০০ এর জায়গায় ১০০০ দিতে পারেন রেজাল্ট দেখে
    for idx, row in df.head(500).iterrows():
        v_id = str(row['video_id']).zfill(3)
        a_file = os.path.join(a_dir, f"{v_id}.npy")
        v_file = os.path.join(v_dir, f"{v_id}.csv")
        
        if os.path.exists(a_file) and os.path.exists(v_file):
            # অডিও লোড
            a_data = np.load(a_file)
            # ভিডিও লোড (পান্ডাস ছাড়া সরাসরি নামপাই দিয়ে দ্রুত লোড)
            v_data = np.genfromtxt(v_file, delimiter=',')
            
            # গড় মান নেওয়া
            a_feats.append(np.mean(a_data, axis=0) if a_data.ndim > 1 else a_data)
            v_feats.append(np.mean(v_data, axis=0) if v_data.ndim > 1 else v_data)
            y.append(row['label'])
            
    return np.array(a_feats), np.array(v_feats), np.array(y)

# ৪. রান করুন
X_audio, X_visual, y_final = load_lite(labels_df, AUDIO_DIR, VISUAL_DIR)
print("-" * 30)
print(f"Successfully Loaded: {len(y_final)} samples")
print(f"Audio Shape: {X_audio.shape[1]}")
print(f"Visual Shape: {X_visual.shape[1]}")

Total Labels: 1823
------------------------------
Successfully Loaded: 500 samples
Audio Shape: 128
Visual Shape: 470
